In [ ]:
import sys
sys.path.append("../src/")

from elektrapy.preprocessing import preprocess_data, get_network_df
from elektrapy.indices import get_redox_index, get_aggregated_potential
from elektrapy.sankey import (
    get_colors,
    get_node_data,
    create_sankey
)

In [ ]:
import os

from ast import literal_eval

import pandas as pd
from pandas.api.types import CategoricalDtype

from matplotlib.colors import to_rgba

import plotly.express as px
import plotly.graph_objects as go
from plotly.colors import hex_to_rgb

from sklearn.preprocessing import minmax_scale

In [ ]:
# Define data folders paths

ROOT_DIR = "../../data/"

METADATA_DIR = os.path.join(
    ROOT_DIR,
    "metadata"
)

DATA_DIR = os.path.join(
    ROOT_DIR,
    "runs",
    "2026-05-31"
)

In [ ]:
# Define output directory for figures based on input path
FIGURES_DIR = os.path.join(
    DATA_DIR,
    "figures"
)
os.makedirs(
    name=FIGURES_DIR,
    exist_ok=True
)

In [ ]:
# Other global variables
COLOR_PALETTE = "magma"
LINK_ALPHA = 0.05

## Metadata

In [ ]:
# Map site name to site ID
metadata_df = pd.merge(
    left=pd.read_csv(
        os.path.join(
            METADATA_DIR,
            "metadata-mags.csv"
        )
    ),
    right=pd.read_csv(
        os.path.join(
            METADATA_DIR,
            "metadata-samples.csv"
        )
    ),
    on=["dataset", "sample_id"],
    how="left"
)

metadata_df = metadata_df.rename(columns={"assembly": "genome_id"})

metadata_df

## Preprocessing

In [ ]:
# Load functional annotation results (from bigecyhmm output)
pathway_df = pd.read_table(
    os.path.join(
        DATA_DIR,
        "bigecyhmm",
        "pathway_presence.tsv"
    )
)
pathway_df.head()

In [ ]:
results_df_genome = preprocess_data(
    df=pathway_df,
    group_var="genome_id"
)
network_df_genome = get_network_df(
    results_df=results_df_genome,
    group_var="genome_id"
)
network_df_genome

In [ ]:
results_df_sample = preprocess_data(
    df=pathway_df,
    group_var="sample_id",
    mapping=dict(zip(metadata_df["genome_id"], metadata_df["sample_id"]))
)
network_df_sample = get_network_df(
    results_df=results_df_sample,
    group_var="sample_id"
)
network_df_sample

In [ ]:
network_df_type = pd.merge(
    left=network_df_sample,
    right=metadata_df[["sample_id", "type"]].drop_duplicates(),
    how="inner",
    on="sample_id"
)

network_df_type["type"] = network_df_type["type"].fillna("unknown")

network_df_type = network_df_type\
    .groupby(["type", "source", "target"], as_index=False)\
    ["value"].sum()

network_df_type

In [ ]:
network_df_depth = pd.merge(
    left=network_df_sample,
    right=metadata_df[["sample_id", "depth"]].drop_duplicates(),
    how="inner",
    on="sample_id"
)

network_df_depth["depth_cat"] = None
network_df_depth.loc[
    network_df_depth["depth"] == 0.0,
    "depth_cat"
] = "surface"
network_df_depth.loc[
    (network_df_depth["depth"] >= 4.5) & (network_df_depth["depth"] < 500),
    "depth_cat"
] = "subsurface (shallow)"
network_df_depth.loc[
    network_df_depth["depth"] >= 500,
    "depth_cat"
] = "subsurface (deep)"
network_df_depth["depth_cat"] = network_df_depth["depth_cat"]\
    .fillna("unknown")

network_df_depth = network_df_depth.drop("depth", axis=1)

network_df_depth = network_df_depth\
    .groupby(["depth_cat", "source", "target"], as_index=False)\
    ["value"].sum()

network_df_depth = network_df_depth[
    network_df_depth["depth_cat"] != "unknown"
]

network_df_depth

## Common parameters for plotting

In [ ]:
cycle_color_map = {
    "hydrogen": "#D8FFC5",
    "carbon": "#2E2910",
    "sulfur": "#FFEA88",
    "nitrogen": "#3874FF",
    "oxygen": "#E63946",
    "iron": "#B34A44",
    "other": "#C7D3C0"
}

node_cycle_map = {
    "H2": "hydrogen",
    "H+": "hydrogen",
    "organic carbon": "carbon",
    "acetate": "carbon",
    "ethanol": "carbon",
    "CH4": "carbon",
    "CO2": "carbon",
    "S0": "sulfur",
    "H2S": "sulfur",
    "SO3": "sulfur",
    "SO4": "sulfur",
    "S2O3": "sulfur",
    "NO2": "nitrogen",
    "NO3": "nitrogen",
    "NH4": "nitrogen",
    "N2O": "nitrogen",
    "NO": "nitrogen",
    "N2": "nitrogen",
    "O2": "oxygen",
    "Fe2+": "iron",
    "Fe3+": "iron",
    "AsO3": "other",
    "AsO4": "other",
    "SeO4": "other"
}

node_colors = pd.DataFrame\
    .from_dict(node_cycle_map, orient="index")\
    .reset_index()\
    .rename(columns={"index": "node", 0: "cycle"})

node_colors["color"] = node_colors["cycle"].map(cycle_color_map)
node_colors = dict(zip(node_colors["node"], node_colors["color"]))

## Electron Flow Diagram (EFD)

In [ ]:
sankey_df = network_df_depth.copy()

sankey_df = sankey_df.rename(columns={"depth_cat": "Depth"})

group_var = "Depth"

In [ ]:
# TODO: check which group_var to use
# X axis: the Redox Tendency Index
rti_df = get_redox_index(
    network_df=network_df_sample,
    group_var="sample_id"
)

# Y axis: the mean of the transformed Eº'
redox_df = get_aggregated_potential(
    redox_df=pd.read_csv("../data/metadata-uhs - potentials.csv"),
    rti_df=rti_df
)

In [ ]:
node_df = pd.merge(
    left=rti_df,
    right=redox_df,
    on="node",
    how="inner"
)

fig = px.scatter(
    data_frame=node_df,
    x="redox_index",
    y="transformed_potential",
    color="redox_index",
    hover_name="node",
    template="plotly_white"
)

# Invert potentials to go from most negative to most positive
fig["layout"]["yaxis"]["autorange"] = "reversed"

fig.show()

In [ ]:
# Manually add minimum and maximum to force the range before scaling
node_df = pd.concat([
    node_df,
    pd.Series({
        "node": "minimum",
        "redox_index": -1,
        "transformed_potential": -1.5
    }).to_frame().T,
    pd.Series({
        "node": "maximum",
        "redox_index": 1,
        "transformed_potential": 1
    }).to_frame().T
])

# Transform axes to fit within the range of [0, 1] for the Sankey
node_df["redox_index_minmax"] = minmax_scale(
    node_df["redox_index"]
)
node_df["transformed_potential_minmax"] = minmax_scale(
    node_df["transformed_potential"]
)

# Drop artificial nodes
node_df = node_df[~node_df["node"].isin(["minimum", "maximum"])].copy()

# Add color
node_df["node_color"] = node_df["node"].map(node_colors)

In [ ]:
categories = CategoricalDtype(
    categories=node_df["node"].unique(),
    ordered=True
)

# Encode as categories for plotting
node_df["node"] = node_df["node"].astype(categories)
sankey_df["source"] = sankey_df["source"].astype(categories)
sankey_df["target"] = sankey_df["target"].astype(categories)

# Force sorting according to categories order for maintaining order in Sankey
# NOTE: missing categories may alter order of colors!
node_df = node_df.sort_values("node")
sankey_df = sankey_df.sort_values("target")

In [ ]:
link_map = {
    "surface": "#FFC349",
    "subsurface (shallow)": "#97DDE9",
    "subsurface (deep)": "#525EA7"
}
sankey_df[f"link_color_{group_var}"] = sankey_df[group_var].map(link_map)

# Modify link opacity
sankey_df[f"link_color_{group_var}"] = sankey_df[f"link_color_{group_var}"]\
    .apply(lambda row: f"rgb{hex_to_rgb(row)}")\
    .apply(lambda row: row.replace("rgb", "rgba"))\
    .apply(lambda row: row.replace(")", f", {LINK_ALPHA})"))

sankey_df.loc[
    (sankey_df["source"] == "H2"),
    f"link_color_{group_var}"
] = sankey_df.loc[
    (sankey_df["source"] == "H2"),
    f"link_color_{group_var}"
].str.replace(f", {LINK_ALPHA})", ", 1)")

In [ ]:
color_var = group_var

fig = go.Figure(
    go.Sankey(
        domain={
            "x": [0.0, 1.0],
            "y": [0.0, 1.0]
        },
        orientation="h",
        arrangement="freeform",

        node={
            # NOTE: use categories here and in the link definition to avoid
            # errors when creating the Sankey (interally sorts the nodes)
            "label": node_df["node"].cat.categories,

            "x": node_df["redox_index_minmax"].values.tolist(),
            "y": node_df["transformed_potential_minmax"].values.tolist(),
            "color": node_df["node_color"].values.tolist(),

            "thickness": 10,
            "pad": 20
        },
        link={
            # HMM hits define enzymes present and, thus, substrates and products
            "label": sankey_df[group_var].values.tolist(),

            "source": sankey_df["source"].cat.codes.tolist(),
            "target": sankey_df["target"].cat.codes.tolist(),
            "color": sankey_df[f"link_color_{group_var}"].values.tolist(),

            # Define number of links by gene copy number and/or abundance
            "value": sankey_df["value"].values.tolist(),

            "arrowlen": 15
        }
    )
)

# Create legends
legend_dataset = [
    go.Scatter(
        mode="lines",
        x=[None],
        y=[None],
        marker=dict(size=10, color=color, symbol="square"),
        name=key,
        legendgroup="color_var",
        legendgrouptitle={
            "text": f"{color_var.capitalize()} (links)"
        }
    )
    for key, color in link_map.items()
]
legend_dataset.extend([
    go.Scatter(
        mode="markers",
        x=[None],
        y=[None],
        marker=dict(size=10, color=color, symbol="square"),
        name=key,
        legendgroup="cycle",
        legendgrouptitle={
            "text": "Cycle (nodes)"
        }
    )
    for key, color in cycle_color_map.items()
])

for trace in legend_dataset:
    fig.add_trace(trace)

fig.update_layout(
    width=1000,
    height=750,
    font=dict(
        size=15,
        # weight="bold",
        family="Arial"
    ),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

# Change to true to plot the axes
fig.update_xaxes(
    visible=False,
    tickvals=[1, 0, -1],
    range=[-1, 1]
)
fig.update_yaxes(
    visible=False,
    tickvals=[-1, -0.5, 0, 0.5, 1, 1.3],
    range=[1.5, -1.5]
)

fig.write_image(
    os.path.join(
        FIGURES_DIR,
        f"efd-ebec.svg"
    ),
    scale=50
)

fig.show()

### Redox index with depth

In [ ]:
network_df_depth = pd.merge(
    left=network_df_sample,
    right=metadata_df[["sample_id", "depth"]].drop_duplicates(),
    how="inner",
    on="sample_id"
)

network_df_depth["depth_cat"] = None
network_df_depth.loc[
    network_df_depth["depth"] == 0.0,
    "depth_cat"
] = "surface"
network_df_depth.loc[
    (network_df_depth["depth"] >= 4.5) & (network_df_depth["depth"] < 500),
    "depth_cat"
] = "subsurface (shallow)"
network_df_depth.loc[
    network_df_depth["depth"] >= 500,
    "depth_cat"
] = "subsurface (deep)"
network_df_depth["depth_cat"] = network_df_depth["depth_cat"]\
    .fillna("unknown")

network_df_depth = network_df_depth.drop("depth", axis=1)

network_df_depth = network_df_depth[
    network_df_depth["depth_cat"] != "unknown"
]

network_df_depth

In [ ]:
node_df_depth = []

for depth in network_df_depth["depth_cat"].unique():

    depth_df = network_df_depth[network_df_depth["depth_cat"] == depth]

    # X axis: the Redox Tendency Index
    rti_df = get_redox_index(
        network_df=depth_df,
        group_var="sample_id"
    )
    rti_df["Depth"] = depth

    node_df_depth.append(rti_df)

node_df_depth = pd.concat(node_df_depth)

fig = px.scatter(
    data_frame=node_df_depth,
    x="redox_index",
    y="node",
    color="Depth",
    color_discrete_map={
        "surface": "#FFC349",
        "subsurface (shallow)": "#97DDE9",
        "subsurface (deep)": "#525EA7"
    },
    category_orders={
        "Depth": [
            "surface",
            "subsurface (shallow)",
            "subsurface (deep)"
        ]
    },
    hover_name="node",
    template="plotly_white"
)

fig.update_layout(
    width=1000,
    height=750,
    font=dict(
        size=15,
        # weight="bold",
        family="Arial"
    ),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

# Invert potentials to go from most negative to most positive
fig["layout"]["yaxis"]["autorange"] = "reversed"

fig.show()